# E05_FoodCollector — REINFORCE, первый метод, обучающий политику напрямую

**Урок:** 2.1 «Policy Gradient» курса [cyber-unity-learn](https://rl-cuber-unity-code.com/)
**Что реализуется:** Monte-Carlo policy gradient (REINFORCE) с baseline.
**Ожидаемый результат:** **≥ 80%** эпизодов с положительной наградой,
то есть агент устойчиво ест больше хорошей еды, чем плохой.
**Время обучения:** около 40 минут на сид (`QUICK_RUN = True` — около двух минут).

## Что здесь принципиально нового

Три вещи сразу, и все три впервые в лаборатории.

**1. Метод обучает политику напрямую.** Табличный Q-learning и DQN учили
ценность, а политику выводили из неё ($\arg\max_a Q$). Здесь параметры политики
двигаются градиентом её собственной целевой функции. Отсюда — вся ветка
policy gradient: A2C, PPO, SAC.

**2. Сеточное наблюдение.** `GridSensor` даёт карту вокруг агента — тензор
$(2, 8, 8)$, по каналу на сорт еды. Это первое наблюдение, которое нельзя
приклеить к вектору: оно уходит в политику отдельным входом и обрабатывается
свёрточной частью сети.

**3. Гибридное действие.** Движение непрерывно (тяга, поворот), ускорение —
дискретный переключатель. Части независимы, поэтому логарифмы складываются:

$$\log \pi(a|s) = \log \pi_{\text{непр}}(a_c|s) + \log \pi_{\text{дискр}}(a_d|s)$$

Ни одна строка самого REINFORCE от этого не меняется — меняется только политика.

## Чего от метода не стоит ждать

REINFORCE делает **один** шаг градиента на всю пачку собранных эпизодов.
При 8 эпизодах на обновление и 8 аренах это обновление
раз в 300 шагов сбора — за весь полный прогон
выходит 800 шагов градиента.
Для сравнения, PPO в `E06` делает их около двадцати тысяч.

Это не недостаток реализации, а свойство метода, и ровно поэтому урок 2.2
существует. Ожидать здесь чисел уровня PPO не следует; ожидать надо
устойчивого различения хорошей и плохой еды.

## Требования

```powershell
python scripts\build_env.py E05_FoodCollector
```

## 1. Окружение

In [ ]:
# --- 1. Окружение -------------------------------------------------------
import platform
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "python" / "labrl").is_dir():
    if REPO_ROOT.parent == REPO_ROOT:
        raise RuntimeError("не найден корень репозитория")
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "python"))

import numpy as np
import torch
import torch.nn as nn
import mlagents_envs

from labrl.utils.seeding import resolve_device, set_global_seed

SEED = 0
set_global_seed(SEED)
DEVICE = resolve_device("auto")

print(f"python        {platform.python_version()}")
print(f"torch         {torch.__version__}  CUDA {torch.version.cuda}")
print(f"mlagents_envs {mlagents_envs.__version__}")
print(f"устройство    {DEVICE}")
print(f"сид           {SEED}")

## 2. Конфигурация

`ReinforceConfig` — свойства **метода**, `ReinforceTrainConfig` — свойства
процесса. Значения совпадают с `configs/E05_FoodCollector__reinforce.yaml`.

Ключевой параметр процесса здесь один: `episodes_per_update`. Обновление
делается по **завершённым эпизодам**, а не по нарезке шагов, — в этом
и состоит «Monte-Carlo» в названии метода: возврат $G_t$ считается
по фактическому эпизоду до конца, без бутстрэпа.

Компромисс прямой: больше эпизодов в пачке — меньше дисперсия оценки
градиента, но меньше самих шагов градиента. Значение 8
выбрано измерением: при 16 полный прогон давал вдвое меньше обновлений
и заметно худший результат при том же бюджете.

Энтропийный бонус (0.01) выше, чем у PPO (0.005): у REINFORCE нет обрезки
шага, и рано схлопнувшаяся политика уже не восстановится — данные для
следующего обновления собирает она же.

In [ ]:
QUICK_RUN = True   # True — быстрая проверка (~2 мин); False — полное обучение (~40 мин)

from labrl.algos.reinforce import ReinforceConfig
from labrl.train.reinforce import ReinforceTrainConfig
from labrl.utils.schedules import LinearSchedule

TOTAL_STEPS = 12_000 if QUICK_RUN else 240_000

ALGO_CFG = ReinforceConfig(
    gamma=0.99,
    learning_rate=0.001,
    value_coef=0.5,           # вес ошибки baseline
    entropy_coef=0.01,        # выше, чем у PPO: обрезки шага здесь нет
    max_grad_norm=0.5,
    normalize_advantage=True,
)

TRAIN_CFG = ReinforceTrainConfig(
    total_steps=TOTAL_STEPS,
    episodes_per_update=8,
    eval_every_steps=max(1, TOTAL_STEPS // 10),   # 24 000 на полном бюджете
    eval_episodes=16,
    # Успех эпизода = положительная награда: съедено больше хорошей еды,
    # чем плохой, с учётом штрафов за время и ускорение.
    success_threshold=0.0,
)

# Расписание НЕ сжимается вместе с сокращённым прогоном (T-10).
LEARNING_RATE = LinearSchedule(start=0.001, end=0.0, decay_steps=240_000)

updates = TOTAL_STEPS // (8 * 300 // 8)
print(f"бюджет: {TOTAL_STEPS} шагов сбора, шагов градиента за прогон: {updates}")

## 3. Подключение среды

In [ ]:
from labrl.envs.registry import get as get_env
from labrl.envs.unity_env import open_unity_env
from labrl.envs.vec_unity_env import VecUnityEnv

ENV_ID = "E05_FoodCollector"
BUILD = get_env(ENV_ID).build_path
if not BUILD.is_file():
    raise FileNotFoundError(f"нет билда {BUILD}\nСоберите: python scripts/build_env.py {ENV_ID}")

handle = open_unity_env(ENV_ID, build_path=BUILD, seed=SEED, time_scale=20.0, no_graphics=True)
vec = VecUnityEnv(handle)
obs = vec.reset()
print(handle.describe())
print(f"\nарен (параллельных сред): {vec.num_envs}")

## 4. Инспекция пространств

Здесь `single_obs_dim` **не применим**: сеточное наблюдение многомерно, и
склеивать его с вектором в один плоский вход значит выбросить всю структуру,
ради которой сетка и заведена. Политика получает наблюдения раздельно.

In [ ]:
GRID_SHAPE = tuple(int(x) for x in vec.obs_shapes[0])     # (каналы, H, W)
VECTOR_DIM = int(vec.obs_shapes[1][0])
CONTINUOUS_DIM = int(vec.action_spec.continuous_size)
BRANCHES = tuple(int(b) for b in vec.action_spec.discrete_branches)

print(f"obs_shapes:        {vec.obs_shapes}")
print(f"сетка:             {GRID_SHAPE}  (канал 0 — хорошая еда, канал 1 — плохая)")
print(f"вектор:            {VECTOR_DIM}")
print(f"непрерывных:       {CONTINUOUS_DIM}")
print(f"дискретные ветки:  {BRANCHES}")

grid = obs[0][0]
print(f"\nв сетке первого агента занятых клеток: "
      f"хорошей еды {int((grid[0] > 0).sum())}, плохой {int((grid[1] > 0).sum())}")
print(f"максимум по сетке: {float(grid.max()):.1f}")

# Значения из строки-контракта ENV_SPEC.md:
# <!-- validator: obs_size=6; discrete_branches=2; continuous_size=2; max_step=1500 -->
assert GRID_SHAPE == (2, 8, 8), f"сетка {GRID_SHAPE} != (2, 8, 8)"
assert VECTOR_DIM == 6, f"вектор {VECTOR_DIM} != 6 (ENV_SPEC.md)"
assert CONTINUOUS_DIM == 2 and BRANCHES == (2,), "пространство действий не гибридное"
# Пустая сетка — типовой отказ: маска слоёв GridSensor по умолчанию нулевая
# и сенсор молча возвращает нули (T-16 в docs/07_TROUBLESHOOTING.md).
assert float(grid.max()) > 0.0, "сетка пуста: проверьте ColliderMask у GridSensorComponent"
print("\nразмерности совпадают с ENV_SPEC.md, сетка не пуста")

## 5. Нейросети

**Политика** состоит из трёх частей:

1. свёрточный ствол по сетке $(2, 8, 8)$ — он и извлекает «где еда»;
2. склейка его выхода с вектором проприоцепции (6 чисел);
3. две головы: среднее непрерывной части и логиты дискретной ветки.

Свёртка здесь уместна ровно потому, что сетка — карта: признак «еда в двух
клетках впереди-слева» имеет один и тот же смысл в любой части тензора,
и разделяемые веса это выражают. Полносвязный слой по 128 входам выучивал бы
каждую клетку отдельно.

**Baseline** — отдельная сеть той же формы, выдающая $V(s)$. Она не участвует
в цели, а только вычитается из возврата. Отключается флагом
`baseline: false` в конфиге — тогда получается «чистый» REINFORCE урока 2.1.

In [ ]:
from labrl.nets.hybrid_policy import GridHybridPolicy, GridValueNetwork


class MyPolicy(GridHybridPolicy):
    """(сетка (B,2,8,8), вектор (B,6)) -> (среднее (B,2), логиты (B,2))."""

    def __init__(self, grid_shape, vector_dim, continuous_dim, branches):
        super().__init__(grid_shape, vector_dim, continuous_dim, branches,
                         conv_channels=(16, 32), hidden_sizes=(128, 128),
                         activation="relu", log_std_init=-0.5)


policy_net = MyPolicy(GRID_SHAPE, VECTOR_DIM, CONTINUOUS_DIM, BRANCHES)
value_net = GridValueNetwork(GRID_SHAPE, VECTOR_DIM, (16, 32), (128, 128), "relu")

with torch.no_grad():
    probe_grid = torch.zeros(3, *GRID_SHAPE)
    probe_vec = torch.zeros(3, VECTOR_DIM)
    mean, logits = policy_net(probe_grid, probe_vec)
    assert mean.shape == (3, CONTINUOUS_DIM), "среднее обязано быть (B, continuous_dim)"
    assert logits.shape == (3, sum(BRANCHES)), "логиты обязаны быть (B, сумма веток)"
    assert value_net(probe_grid, probe_vec).shape == (3, 1), "baseline обязан выдавать (B, 1)"

print(policy_net)
print(f"\nпараметров политики: {sum(p.numel() for p in policy_net.parameters()):,}")
print(f"параметров baseline: {sum(p.numel() for p in value_net.parameters()):,}")

## 6. Теория и математика метода

Теорема о градиенте политики:

$$\nabla_\theta J(\theta) = \mathbb{E}\Bigl[\sum_t \nabla_\theta
\log \pi_\theta(a_t|s_t)\,G_t\Bigr], \qquad G_t = \sum_{k \ge t}\gamma^{k-t} r_k$$

Оценка несмещённая, но крайне шумная: множитель $G_t$ содержит **всё**, что
случилось после шага $t$, в том числе никак с ним не связанное.

Лечение — baseline:

$$\nabla_\theta J = \mathbb{E}\Bigl[\sum_t \nabla_\theta
\log \pi_\theta(a_t|s_t)\,(G_t - b(s_t))\Bigr]$$

**Почему это законно.** Для любой функции состояния

$$\mathbb{E}_{a\sim\pi}\bigl[\nabla_\theta \log \pi_\theta(a|s)\,b(s)\bigr]
= b(s)\,\nabla_\theta \sum_a \pi_\theta(a|s) = b(s)\,\nabla_\theta 1 = 0$$

Математическое ожидание не меняется, дисперсия падает. Лучший выбор
$b(s) = V^\pi(s)$ — и в скобках оказывается оценка преимущества. Отсюда
один шаг до A2C: заменить Monte-Carlo возврат бутстрэпом.

**Обрыв по времени — не провал.** Все эпизоды этой среды кончаются
по `MaxStep`, и возврат последнего шага считается с бутстрэпом
$G_T = V(s_T)$. Без этого метод выучил бы, что конец отсчёта времени
равносилен провалу (требование 8.3).

Ниже — исходный код обновления, тот самый, что выполняется при обучении.

In [ ]:
import inspect

from labrl.algos.reinforce import REINFORCE, discounted_returns

print(inspect.getsource(REINFORCE.update))
print(inspect.getsource(discounted_returns))

## 7. Цикл обучения

Цикл `labrl.train.reinforce.train_reinforce` отличается от on-policy цикла
одним: он накапливает **завершённые** эпизоды по аренам и вызывает `update`,
только когда их набралось `episodes_per_update`. Незавершённые эпизоды
в обновление не попадают — Monte-Carlo возврат для них ещё не определён.

In [ ]:
from labrl.logging.run_dir import create_run_dir
from labrl.logging.tb_logger import TBLogger
from labrl.train.reinforce import train_reinforce

algo = REINFORCE(policy_net, value_net, ALGO_CFG, device=DEVICE)
run = create_run_dir(ENV_ID, "reinforce", SEED)
run.write_env_info({**handle.env_info(), "num_envs": vec.num_envs, "seed": SEED,
                    "total_steps": TOTAL_STEPS, "quick": QUICK_RUN})
logger = TBLogger(run.tb)

print(f"каталог прогона: {run.root}\n")

result = train_reinforce(
    vec=vec, algo=algo, logger=logger, cfg=TRAIN_CFG, seed=SEED,
    lr_schedule=LEARNING_RATE,
    on_eval=lambda step, r, s: print(f"  шаг {step:>7}: награда {r:>7.3f}, успешных {s:.0%}"),
)

missing = logger.missing_required_tags()
logger.dump_metrics(run.metrics_json, extra={"seed": SEED, "missing_required_tags": list(missing)})
logger.close()

print(f"\nэпизодов: {len(result.episode_returns)}, обновлений: {result.updates}")
print(f"время: {result.wall_time:.1f} с")
print(f"обязательные теги схемы 11.2 без данных: {missing or 'нет'}")

## 8. Оценка и кривая обучения

Мера качества — доля эпизодов с положительной наградой: агент съел больше
хорошей еды, чем плохой.

Порог задан именно по доле, а не по величине награды, и это измеренное
решение. У награды здесь **нет потолка**: еда переставляется, и за 300
решений её можно собрать сколько успеешь. Величина поэтому шумит от одного
батча оценки к другому — в зондирующем прогоне соседние оценки одной и той же
обучающейся политики дали 6.90, 3.24 и 6.80 при доле успешных 0.75, 0.75
и 1.00. Порог по такому числу проверял бы удачу батча
(T-17 в `docs/07_TROUBLESHOOTING.md`).

In [ ]:
import matplotlib
matplotlib.use("Agg")   # ноутбук должен исполняться и без графического бэкенда
import matplotlib.pyplot as plt

returns = np.array(result.episode_returns)
window = max(1, len(returns) // 50)
smoothed = np.convolve(returns, np.ones(window) / window, mode="valid")

fig, axes = plt.subplots(1, 3, figsize=(15, 3.6))

axes[0].plot(smoothed)
axes[0].axhline(0.0, color="k", lw=0.8)
axes[0].set_title(f"Награда за эпизод (скользящее среднее по {window})")
axes[0].set_xlabel("эпизод"); axes[0].set_ylabel("награда"); axes[0].grid(alpha=0.3)

steps = [s for s, _, _ in result.eval_history]
rewards = [r for _, r, _ in result.eval_history]
success = [s for _, _, s in result.eval_history]

axes[1].plot(steps, success, marker="o", color="tab:green")
axes[1].axhline(0.8, color="k", ls="--", lw=1, label="критерий приёмки")
axes[1].set_ylim(-0.05, 1.05)
axes[1].set_title("Eval/Success Rate — доля эпизодов с положительной наградой")
axes[1].set_xlabel("шаг сбора"); axes[1].grid(alpha=0.3); axes[1].legend()

axes[2].plot(steps, rewards, marker="o", color="tab:orange")
axes[2].axhline(0.0, color="k", lw=0.8)
axes[2].set_title("Eval/Mean Reward")
axes[2].set_xlabel("шаг сбора"); axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"итоговая оценка: успешных {result.last_eval_success:.0%}, "
      f"награда {result.last_eval_reward:.3f}")

THRESHOLD = 0.8   # блок success_criteria конфига, метрика Eval/Success Rate

if QUICK_RUN:
    print(f"быстрый прогон: критерий приёмки ({THRESHOLD:.0%}) не проверяется — "
          "проверялась работоспособность конвейера")
else:
    assert result.last_eval_success >= THRESHOLD, (
        f"не достигнут критерий приёмки: {result.last_eval_success:.0%} < {THRESHOLD:.0%}")
    print(f"критерий приёмки достигнут: {result.last_eval_success:.0%} >= {THRESHOLD:.0%}")

## 9. Экспорт в ONNX и верификация

Гибридное действие даёт в графе **обе** группы выходов: `continuous_actions`
и `discrete_actions`. Дискретная часть выходит **индексом** выбранной ветки,
а не one-hot (`docs/04_ONNX_CONTRACT.md`, §4.3), и `*_output_shape` для неё
описывает размеры веток.

Входов тоже два — сетка и вектор, в том же порядке, в каком их отдаёт среда.
Baseline не экспортируется: он существовал ради снижения дисперсии.

In [ ]:
import onnxruntime as ort

from labrl.export.onnx_export import ActionSpecLite, export_policy_to_onnx
from labrl.export.onnx_verify import verify_onnx_model

# Наблюдения из реального распределения среды (требование 10.5), а не шум.
collected = [[] for _ in vec.obs_shapes]
obs_now = vec.reset()
while sum(len(c) for c in collected[0]) < 64:
    for sensor, batch in enumerate(obs_now):
        collected[sensor].append(batch.copy())
    obs_now = vec.step(algo.deterministic_action(obs_now)).obs
sample = [np.concatenate(parts, axis=0)[:64] for parts in collected]

spec = ActionSpecLite(continuous_size=CONTINUOUS_DIM, discrete_branches=BRANCHES)
policy = algo.policy_module()
# Многомерное наблюдение нельзя склеивать: политика получает входы раздельно.
onnx_path = export_policy_to_onnx(policy, spec, vec.obs_shapes, run.onnx / "policy.onnx",
                                  obs_combiner=None)

verification = verify_onnx_model(onnx_path, policy, spec, vec.obs_shapes, sample_obs=sample,
                                 obs_combiner=None)
print(verification.report())
verification.raise_if_failed()

# Python и граф обязаны давать одно и то же действие — обе его части.
session = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
feeds = {"obs_0": sample[0], "obs_1": sample[1]}
graph_continuous = session.run(["deterministic_continuous_actions"], feeds)[0]
graph_discrete = session.run(["deterministic_discrete_actions"], feeds)[0]
python_action = algo.deterministic_action(sample)

gap = float(np.abs(graph_continuous - python_action[:, :CONTINUOUS_DIM]).max())
mismatch = int((graph_discrete.reshape(len(sample[0]), -1)
                != python_action[:, CONTINUOUS_DIM:].astype(graph_discrete.dtype)).sum())
print(f"\nmax|ONNX - Python| по непрерывной части: {gap:.3e}")
print(f"расхождений по дискретной ветке: {mismatch} из {len(sample[0])}")
assert gap < 1e-5 and mismatch == 0, "оценка в Python и граф расходятся"

if not QUICK_RUN:
    # Модель проекта обновляется только полным прогоном (T-14).
    model_dest = (REPO_ROOT / "unity" / "MLAgentsLab" / "Assets" / "Envs" / ENV_ID /
                  "Models" / f"{ENV_ID}_reinforce.onnx")
    model_dest.parent.mkdir(parents=True, exist_ok=True)
    model_dest.write_bytes(onnx_path.read_bytes())
    print(f"модель скопирована в проект Unity: {model_dest.relative_to(REPO_ROOT)}")
else:
    print("быстрый прогон: модель в проект Unity НЕ копируется (T-14)")

vec.close()

## 10. Возврат в Unity

### Автоматически (то, чем проверяется приёмка по 10.6)

```powershell
python scripts\check_inference.py --config configs\E05_FoodCollector__reinforce.yaml --python-reward <оценка из раздела 8>
```

### Вручную (чтобы увидеть глазами)

1. Открыть `unity/MLAgentsLab` в Unity 6000.5.8f1.
2. Открыть `Assets/Envs/E05_FoodCollector/Scenes/E05_FoodCollector.unity`.
3. Выделить модель в Project → **Tools → RL → Check ONNX Contract**.
4. У агента `Collector`: `Model` = импортированная модель,
   `Behavior Type` = `Inference Only`, галочка `Deterministic Inference`.
5. Play: собиратель едет к зелёным шарам и объезжает красные.

### Если поведение в Unity другое

Первое, что проверяют в этой среде, — **сетка**: если `ColliderMask`
у `GridSensorComponent` нулевая, сенсор молча отдаёт нули и агент действует
вслепую (T-16). Второе — порядок входов: `obs_0` сетка, `obs_1` вектор.

### Ручная проверка среды с клавиатуры

`Behavior Type` = `Heuristic Only`, **↑/↓** — тяга, **←/→** — поворот,
**Пробел** — ускорение.